# Tarea 2 – Parte 1: Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿Pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

## 1.1. Configuración inicial

Registrar en la primera celda Markdown la fecha de extracción, las fuentes y licencias, y las versiones de PaddleOCR y del modelo ASR (usen los entornos de las sesiones: uv sync --group docai y uv sync --group audio).

- **Fecha de extracción:** 22 de octubre de 2026
- **Fuentes y licencias:** (por completar)
- **Versiones:** Python 3.12.14 · PaddleOCR 3.7.0 (PaddlePaddle 3.3.1, modelos PP-OCRv6 medium, `lang="es"`) · faster-whisper 1.2.1 (modelo `small`, int8, CPU) · pyannote-audio 4.0.7 · torch 2.14.1

In [5]:
import os
import json
import time
from importlib.metadata import version
from pathlib import Path

# Se evita que PaddleOCR revise la conexión con cada servidor de modelos (va antes de importarlo)
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import torch  # Se importa antes que paddleocr: en Windows, al revés, torch falla (WinError 127)
import pandas as pd
from paddleocr import PaddleOCR
from faster_whisper import WhisperModel

for paquete in ["paddleocr", "paddlepaddle", "faster-whisper", "pyannote-audio", "torch"]:
    print(f"{paquete}: {version(paquete)}")


paddleocr: 3.7.0
paddlepaddle: 3.3.1
faster-whisper: 1.2.1
pyannote-audio: 4.0.7
torch: 2.14.1


In [4]:
# Se crea el lector OCR en español (modelos PP-OCRv6).
# enable_mkldnn=False evita el error de oneDNN en procesadores Intel.
# El clasificador de orientación endereza las fotos giradas 90°, y el texto se busca en una copia
# de máximo 1600 px para que las fotos de 12 MP no tarden más de 10 minutos cada una en esta laptop.
ocr = PaddleOCR(
    lang="es",
    enable_mkldnn=False,
    use_doc_orientation_classify=True,    # se endereza la página si está girada
    use_doc_unwarping=False,              # no se "aplana" la hoja
    use_textline_orientation=False,       # no se gira cada línea
    text_det_limit_type="max",
    text_det_limit_side_len=1600,         # lado mayor de la copia donde se busca el texto
)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [3]:
# Se carga faster-whisper "small" en CPU con int8 (la misma configuración de la línea base de la Parte 2)
modelo_asr = WhisperModel("small", device="cpu", compute_type="int8")

## 1.2. OCR (Sesión 10):

Extraer texto de los 25 documentos con PaddleOCR (impresos, fotos y manuscritos),guardando respaldo crudo y recortes en datos/raw/documentos/. Documentar parámetros (idioma,preprocesamiento).

A continuación, se describen los documentos selecciones:

1. 8 imágenes imprenta: Corresponde al documento de mi titulación.
2. 8 fotos de un documento imprenta: Corresponde al docuemento de SERVIR "Estudio de gobiernos locales 2016: Principales hallazgos sobre los puestos con los que cuentan y las personas que los ocupan" (https://repositorio.servir.gob.pe/item/025583e3-60dc-4f56-91d3-9160138f24b8). En la página 4, indica que se puede usar este documento con fines de estudios privados, siempre que se reconozca la autoría de SERVIR.
3. 9 fotos de manuscritos: Corresponde a mis apuntes de clase de estadística, matemática y cerámica.

**Parámetros del OCR:** español (`lang="es"`, modelos PP-OCRv6 medium); clasificador de orientación activado para enderezar las fotos giradas 90°; el texto se busca en una copia de máximo 1600 px, porque en esta laptop (CPU de 2 núcleos) cada foto de 12 MP tardaba de 9 a 13 minutos y así tarda cerca de 3; oneDNN desactivado por un error de PaddlePaddle en procesadores Intel. No se aplica otro preprocesamiento. Los respaldos crudos (JSON con texto, cajas y confianza, e imagen con las cajas detectadas) se guardan en `datos/raw/documentos/ocr/`.

In [6]:
# Se listan los 25 documentos y se asigna su tipo según el número: 001–008 impresos, 009–016 fotos y 017–025 manuscritos
CARPETA_DOCS = Path("datos/raw/documentos")
CARPETA_OCR = CARPETA_DOCS / "ocr"          # respaldo crudo del OCR
CARPETA_OCR.mkdir(exist_ok=True)

def tipo_documento(numero):
    if numero <= 8:
        return "impreso"
    elif numero <= 16:
        return "foto"
    return "manuscrito"

archivos = sorted(CARPETA_DOCS.glob("doc_*.*"))
documentos = pd.DataFrame({"id": [f.stem for f in archivos], "archivo": [f.as_posix() for f in archivos]})
documentos["tipo"] = documentos["id"].str[-3:].astype(int).apply(tipo_documento)
documentos["tipo"].value_counts()

tipo
manuscrito    9
impreso       8
foto          8
Name: count, dtype: int64

**Excepción:** en la primera ejecución, el clasificador de orientación giró por error 180° la foto `doc_009` (inclinada y con una franja gris oscura arriba): su imagen enderezada quedó de cabeza y el texto salió ilegible (confianza media 0,64). Por eso, ese documento se procesa sin el clasificador; el resto de la configuración es la misma.

In [8]:
# Segundo lector, igual al primero pero sin el clasificador de orientación, solo para doc_009
SIN_ORIENTACION = ["doc_009"]
ocr_sin_orientacion = PaddleOCR(
    lang="es",
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    text_det_limit_type="max",
    text_det_limit_side_len=1600,
)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LENOVO\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [12]:
# Se aplica el OCR a cada documento y se guarda su respaldo crudo (JSON e imágenes) en datos/raw/documentos/ocr/.
# Si un documento ya tiene su JSON, se lee de ahí y no se vuelve a procesar: así, al ejecutar de nuevo el notebook,
# no se repite la hora de OCR.
textos = []
for archivo in documentos["archivo"]:
    respaldo = CARPETA_OCR / f"{Path(archivo).stem}_res.json"
    if respaldo.exists():
        print(f"{Path(archivo).name}: se lee el respaldo")
    else:
        inicio = time.perf_counter()
        lector = ocr_sin_orientacion if Path(archivo).stem in SIN_ORIENTACION else ocr
        resultado = lector.predict(archivo)[0]
        resultado.save_to_json(CARPETA_OCR)
        resultado.save_to_img(CARPETA_OCR)
        print(f"{Path(archivo).name}: {time.perf_counter() - inicio:.0f} s")
    with open(respaldo, encoding="utf-8") as f:
        textos.append("\n".join(json.load(f)["rec_texts"]))

documentos["texto_ocr"] = textos
documentos["modelo_ocr"] = "PaddleOCR 3.7.0 (PP-OCRv6 medium, lang=es, orientación activada, detección máx. 1600 px)"
documentos.loc[documentos["id"].isin(SIN_ORIENTACION), "modelo_ocr"] = "PaddleOCR 3.7.0 (PP-OCRv6 medium, lang=es, orientación desactivada, detección máx. 1600 px)"

doc_001.png: se lee el respaldo
doc_002.png: se lee el respaldo
doc_003.png: se lee el respaldo
doc_004.png: se lee el respaldo
doc_005.png: se lee el respaldo
doc_006.png: se lee el respaldo
doc_007.png: se lee el respaldo
doc_008.png: se lee el respaldo
doc_009.jpg: 191 s
doc_010.jpg: se lee el respaldo
doc_011.jpg: se lee el respaldo
doc_012.jpg: se lee el respaldo
doc_013.jpg: se lee el respaldo
doc_014.jpg: se lee el respaldo
doc_015.jpg: se lee el respaldo
doc_016.jpg: se lee el respaldo
doc_017.jpg: se lee el respaldo
doc_018.jpg: se lee el respaldo
doc_019.jpg: se lee el respaldo
doc_020.jpg: se lee el respaldo
doc_021.jpg: se lee el respaldo
doc_022.jpg: se lee el respaldo
doc_023.jpg: se lee el respaldo
doc_024.jpg: se lee el respaldo
doc_025.jpg: se lee el respaldo
